<a href="https://colab.research.google.com/github/sleep-is-best/Ai/blob/main/Untitled1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers[sentencepiece] protobuf

In [2]:
!pip install -q transformers[sentencepiece] protobuf

### 🏺 نظام التدريب الشامل للأبجدية الميسينية (Linear B)

In [2]:
import pandas as pd
import torch
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from PIL import Image
import cv2
import os

# تحديث الأبجدية الميسينية الشاملة بناءً على طلب المستخدم
full_mycenaean_mapping = {
    '𐀀': 'a', '𐀁': 'e', '𐀂': 'i', '𐀃': 'o', '𐀄': 'u',
    '𐀅': 'da', '𐀆': 'de', '𐀇': 'di', '𐀈': 'do', '𐀉': 'du',
    '𐀊': 'ja', '𐀋': 'je', '𐀍': 'jo', '𐀎': 'ju',
    '𐀏': 'ka', '𐀐': 'ke', '𐀑': 'ki', '𐀒': 'ko', '𐀓': 'ku',
    '𐀔': 'ma', '𐀕': 'me', '𐀖': 'mi', '𐀗': 'mo', '𐀘': 'mu',
    '𐀙': 'na', '𐀚': 'ne', '𐀛': 'ni', '𐀜': 'no', '𐀝': 'nu',
    '𐀞': 'pa', '𐀟': 'pe', '𐀠': 'pi', '𐀡': 'po', '𐀢': 'pu',
    '𐀣': 'qa', '𐀤': 'qe', '𐀥': 'qi', '𐀦': 'qo',
    '𐀨': 'ra', '𐀩': 're', '𐀪': 'ri', '𐀫': 'ro', '𐀬': 'ru',
    '𐀭': 'sa', '𐀮': 'se', '𐀯': 'si', '𐀰': 'so', '𐀱': 'su',
    '𐀲': 'ta', '𐀳': 'te', '𐀴': 'ti', '𐀵': 'to', '𐀶': 'tu',
    '𐀷': 'wa', '𐀸': 'we', '𐀹': 'wi', '𐀺': 'wo',
    '𐀼': 'za', '𐀽': 'ze', '𐀿': 'zo',
    '𐁀': 'a2', '𐁁': 'a3', '𐁂': 'au', '𐁃': 'dwe', '𐁄': 'dwo',
    '𐁅': 'nwa', '𐁇': 'pte', '𐁆': 'pu2', '𐁈': 'ra2', '𐁉': 'ra3',
    '𐁊': 'ro2', '𐁋': 'ta2', '𐁌': 'twe', '𐁍': 'two'
}

# إضافة الرموز الفاصلة
full_mycenaean_mapping['𐄁'] = 'separator'

data = {
    'file_name': ['اثار للغة الميسينية.png'],
    'text': ['𐀗𐀛𐄁𐀀𐀸𐀆𐄁𐀳𐀀𐄁𐀟𐀩𐀷𐀆𐀃𐀍𐄁𐀀𐀑𐀩ဃa𐄁']
}
df_labels = pd.DataFrame(data)
display(df_labels)
print(f"Total Characters mapped: {len(full_mycenaean_mapping)}")

,file_name,text
0,اثار للغة الميسينية.png,𐀗𐀛𐄁𐀀𐀸𐀆𐄁𐀳𐀀𐄁𐀟𐀩𐀷𐀆𐀃𐀍𐄁𐀀𐀑𐀩ဃa𐄁


Total Characters mapped: 75


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class MycenaeanDataset(Dataset):
    def __init__(self, df, processor):
        self.df = df
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = os.path.join('/content', self.df['file_name'][idx])
        image = cv2.imread(img_path)
        if image is None: raise FileNotFoundError(f"Image not found at {img_path}")

        # المعالجة المسبقة لتحسين التعرف
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        _, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY_INV)

        pixel_values = self.processor(Image.fromarray(thresh).convert("RGB"), return_tensors="pt").pixel_values
        labels = self.processor.tokenizer(self.df['text'][idx], padding="max_length", max_length=128).input_ids
        labels = [label if label != self.processor.tokenizer.pad_token_id else -100 for label in labels]

        return {"pixel_values": pixel_values.squeeze(), "labels": torch.tensor(labels)}

In [3]:
import sentencepiece
import transformers
from transformers import TrOCRProcessor, VisionEncoderDecoderModel, RobertaTokenizer, ViTImageProcessor
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from PIL import Image
import torch
import cv2
import os
import pandas as pd

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# الأبجدية الميسينية الأصلية (Linear B Unicode)
full_mycenaean_mapping = {
    '𐀀': 'a', '𐀁': 'e', '𐀂': 'i', '𐀃': 'o', '𐀄': 'u',
    '𐀅': 'da', '𐀆': 'de', '𐀇': 'di', '𐀈': 'do', '𐀉': 'du',
    '𐀊': 'ja', '𐀋': 'je', '𐀍': 'jo', '𐀎': 'ju',
    '𐀏': 'ka', '𐀐': 'ke', '𐀑': 'ki', '𐀒': 'ko', '𐀓': 'ku',
    '𐀔': 'ma', '𐀕': 'me', '𐀖': 'mi', '𐀗': 'mo', '𐀘': 'mu',
    '𐀙': 'na', '𐀚': 'ne', '𐀛': 'ni', '𐀜': 'no', '𐀝': 'nu',
    '𐀞': 'pa', '𐀟': 'pe', '𐀠': 'pi', '𐀡': 'po', '𐀢': 'pu',
    '𐀣': 'qa', '𐀤': 'qe', '𐀥': 'qi', '𐀦': 'qo',
    '𐀨': 'ra', '𐀩': 're', '𐀪': 'ri', '𐀫': 'ro', '𐀬': 'ru',
    '𐀭': 'sa', '𐀮': 'se', '𐀯': 'si', '𐀰': 'so', '𐀱': 'su',
    '𐀲': 'ta', '𐀳': 'te', '𐀴': 'ti', '𐀵': 'to', '𐀶': 'tu',
    '𐀷': 'wa', '𐀸': 'we', '𐀹': 'wi', '𐀺': 'wo',
    '𐀼': 'za', '𐀽': 'ze', '𐀿': 'zo',
    '𐁀': 'a2', '𐁁': 'a3', '𐁂': 'au', '𐁃': 'dwe', '𐁄': 'dwo',
    '𐁅': 'nwa', '𐁇': 'pte', '𐁆': 'pu2', '𐁈': 'ra2', '𐁉': 'ra3',
    '𐁊': 'ro2', '𐁋': 'ta2', '𐁌': 'twe', '𐁍': 'two', '𐄁': 'separator'
}

data = {
    'file_name': ['اثار للغة الميسينية.png'],
    'text': ['𐀗𐀛𐄁𐀀𐀸𐀆𐄁𐀳𐀀𐄁𐀟𐀩𐀷𐀆𐀃𐀍𐄁𐀀𐀑𐀩𐀺𐄁']
}
df_labels = pd.DataFrame(data)

class MycenaeanDataset(Dataset):
    def __init__(self, df, processor):
        self.df = df
        self.processor = processor
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        img_path = os.path.join('/content', self.df['file_name'][idx])
        image = cv2.imread(img_path)
        if image is None: raise FileNotFoundError(f'Image not found at {img_path}')
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        thresh = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11, 2)
        pixel_values = self.processor(Image.fromarray(thresh).convert('RGB'), return_tensors='pt').pixel_values
        labels = self.processor.tokenizer(self.df['text'][idx], padding='max_length', max_length=128).input_ids
        labels = [label if label != self.processor.tokenizer.pad_token_id else -100 for label in labels]
        return {'pixel_values': pixel_values.squeeze(), 'labels': torch.tensor(labels)}

tokenizer = RobertaTokenizer.from_pretrained('microsoft/trocr-base-handwritten', use_fast=False)
image_processor = ViTImageProcessor.from_pretrained('microsoft/trocr-base-handwritten')
new_tokens = list(full_mycenaean_mapping.keys())
tokenizer.add_tokens(new_tokens)

processor = TrOCRProcessor(image_processor=image_processor, tokenizer=tokenizer)
model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-base-handwritten').to(device)
model.decoder.resize_token_embeddings(len(tokenizer))
model.config.vocab_size = len(tokenizer)
model.config.decoder_start_token_id = tokenizer.cls_token_id
model.config.pad_token_id = tokenizer.pad_token_id

dataset = MycenaeanDataset(df_labels, processor)
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)
optimizer = AdamW(model.parameters(), lr=1e-4)

print('Starting training with authentic Linear B Unicode symbols...')
model.train()
for epoch in range(50):
    total_loss = 0
    for batch in dataloader:
        optimizer.zero_grad()
        outputs = model(pixel_values=batch['pixel_values'].to(device), labels=batch['labels'].to(device))
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}/50 - Loss: {total_loss:.4f}')
print('Training complete!')

Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.bias   | MISSING | 
encoder.pooler.dense.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Starting training with authentic Linear B Unicode symbols...
Epoch 10/50 - Loss: 9.7550
Epoch 20/50 - Loss: 7.5374
Epoch 30/50 - Loss: 5.0358
Epoch 40/50 - Loss: 2.8852
Epoch 50/50 - Loss: 2.6669
Training complete!


In [4]:
model.eval()
image_path = '/content/اثار للغة الميسينية.png'
image = cv2.imread(image_path)
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
thresh = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11, 2)
pixel_values = processor(Image.fromarray(thresh).convert('RGB'), return_tensors='pt').pixel_values.to(device)

with torch.no_grad():
    generated_ids = model.generate(
        pixel_values,
        max_new_tokens=50,
        num_beams=5,
        repetition_penalty=3.0,
        length_penalty=1.0,
        early_stopping=True
    )
    generated_text = processor.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

print(f'Detected text (raw): {generated_text}')

decoded_output = []
for char in generated_text:
    if char in full_mycenaean_mapping:
        decoded_output.append(full_mycenaean_mapping[char])
    else:
        decoded_output.append(f'[{char}]')

print(f'Phonetic Translation: {"-".join(decoded_output)}')

Detected text (raw): 𐄁𐀩𐀀𐀆𐀗𐀸𐀛𐀳𐀑𐀺𐀟𐀃𐀷𐀍𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁
Phonetic Translation: separator-re-a-de-mo-we-ni-te-ki-wo-pe-o-wa-jo-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator
